# Hospital Patient Data Pipeline
### Python Analytics Layer — Hospital Management System Project

**What this notebook does:**  
Sits on top of the existing MySQL hospital database and builds a complete analytics pipeline:
- Connects to the database (MySQL live **or** offline CSV mode)
- Loads all 7 tables into Pandas DataFrames
- Performs EDA and data quality checks
- Generates key business insights
- Produces and saves 5 publication-ready charts

**Source project:** `Hospital-Management-System-SQL` (MySQL)  
**Data mode:** Configurable — see Section 1  

---

## Section 0 — Setup & Imports

In [ ]:
# Standard library
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Ensure the python_analytics directory is on the path
# (needed when running the notebook from the repo root)
PIPELINE_DIR = os.path.dirname(os.path.abspath('__file__'))
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

# Third-party
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pipeline modules
import config
from data_loader import load_all
from eda import run_full_eda
import insights
import visualizations as viz

%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('All imports successful.')
print(f'   Pandas  : {pd.__version__}')
print(f'   Seaborn : {sns.__version__}')

✅  All imports successful.
   Pandas  : 2.2.3
   Seaborn : 0.13.2


## Section 1 — Configuration

**Toggle `DATA_SOURCE` here before running:**

| Value | Behavior |
|---|---|
| `"csv"` (default) | Load from pre-exported CSVs in `/data/`. No MySQL needed. |
| `"mysql"` | Connect live to MySQL using credentials in `config.py`. |

In [3]:
# ── Data source toggle ─────────────────────────────────────────────────────
# Change this to 'mysql' if you have a running MySQL server.
config.DATA_SOURCE = 'csv'   # <── edit here

# ── If using MySQL, update credentials in config.py ────────────────────────
# config.MYSQL_CONFIG = {
#     'host': 'localhost',
#     'port': 3306,
#     'user': 'root',
#     'password': 'your_password',
#     'database': 'HOSPITAL_MANAGMENT_SYSTEM',
# }

print(f'DATA_SOURCE = "{config.DATA_SOURCE}"')
print(f'VISUALS_DIR = "{config.VISUALS_DIR}"')

DATA_SOURCE = "csv"
VISUALS_DIR = "C:\Users\nikhi\Desktop\ShowTime\hospitalManagementSQL\python_analytics\visuals"


## Section 2 — Data Loading

Load all 7 tables and build the merged master DataFrame.

In [ ]:
# Load all raw tables + build master merged DataFrame
dfs, master = load_all()

print('\n── Raw tables loaded ─────────────────────────────────────')
for name, df in dfs.items():
    print(f'  {name:<20} {df.shape[0]:>3} rows × {df.shape[1]} cols')

In [ ]:
# Preview the master (merged) DataFrame
print(f'Master DataFrame: {master.shape[0]} rows × {master.shape[1]} columns')
print(f'Columns: {list(master.columns)}\n')
master.head(5)

## Section 3 — EDA & Data Quality Report

Check for nulls, duplicates, formatting issues, and type mismatches.

In [ ]:
eda_results = run_full_eda(dfs)

In [ ]:
# Use the cleaned patient DataFrame going forward
patient_clean = eda_results['patient_cleaned']

print('Cleaned patient DataFrame columns:')
print(list(patient_clean.columns))
patient_clean[['patient_id', 'name', 'surname', 'gender', 'street_type', 'phone']].head(8)

## Section 4 — Insight: Diagnosis Frequency

Which diagnoses appear most often? Broken down overall, by physician, and by department.

In [ ]:
# ── 4a. Top diagnoses overall ──────────────────────────────────────────────
diag_freq = insights.diagnosis_frequency(master, top_n=10)

In [ ]:
# ── 4b. Diagnoses per physician ────────────────────────────────────────────
diag_by_phy = insights.diagnosis_by_physician(master)

In [ ]:
# ── 4c. Diagnoses per department ───────────────────────────────────────────
diag_by_dept = insights.diagnosis_by_department(master)

## Section 5 — Insight: Physician Workload

Two metrics: (1) number of **diagnoses** each physician handled, and (2) number of patients under their **primary care**.

In [ ]:
# Patients per primary physician
primary_counts = insights.physician_patient_count(dfs['patient'], dfs['physician'])

In [ ]:
# Combined workload: diagnoses + primary patients
workload = insights.physician_workload_combined(master, dfs['patient'], dfs['physician'])

## Section 6 — Insight: Procedure Costs

>  **Schema note:** The `procedures` table is a standalone cost **catalog**. No `patient_procedure` junction table exists in the schema, so we cannot compute costs incurred per patient or department. We analyze the catalog statistically.


In [ ]:
# Key statistics
cost_stats = insights.procedure_cost_stats(dfs['procedures'])

In [ ]:
# Ranked cost catalog
cost_catalog = insights.procedure_cost_by_name(dfs['procedures'])
cost_catalog

## Section 7 — Insight: Patient Demographics

In [ ]:
# Gender distribution
gender_dist = insights.gender_distribution(patient_clean)

In [ ]:
# Geographic proxy (street type distribution)
street_dist = insights.street_type_distribution(patient_clean)

In [ ]:
# Nurse registration breakdown
nurse_stats = insights.nurse_registration_stats(dfs['nurse'])

## Section 8 — Schema Limitation Notes

Documented limitations of the existing MySQL schema that prevent certain insights.

In [ ]:
insights.print_schema_limitations()

## Section 9 — Visualizations

All charts are saved to `/visuals/` as PNG files (suitable for embedding in the README).

In [ ]:
# ── Chart 1: Top 10 Diagnoses ──────────────────────────────────────────────
fig1 = viz.plot_top_diagnoses(diag_freq, top_n=10)
plt.show()

In [ ]:
# ── Chart 2: Physician Workload ────────────────────────────────────────────
fig2 = viz.plot_physician_workload(workload, top_n=15)
plt.show()

In [ ]:
# ── Chart 3: Procedure Cost Distribution ──────────────────────────────────
fig3 = viz.plot_procedure_cost_distribution(dfs['procedures'])
plt.show()

In [ ]:
# ── Chart 4: Patient Gender Distribution ──────────────────────────────────
fig4 = viz.plot_gender_distribution(gender_dist)
plt.show()

In [ ]:
# ── Chart 5 (Bonus): Diagnoses by Department ──────────────────────────────
fig5 = viz.plot_diagnoses_by_department(diag_by_dept)
plt.show()

In [ ]:
# Confirm all PNGs were saved
import os
print('Saved charts in:', config.VISUALS_DIR)
for f in sorted(os.listdir(config.VISUALS_DIR)):
    if f.endswith('.png'):
        size_kb = os.path.getsize(os.path.join(config.VISUALS_DIR, f)) // 1024
        print(f'  📊  {f}  ({size_kb} KB)')